# Chapitre 8 — Jointures et concaténation

## Le problème

Une base relationnelle répartit l'information entre plusieurs tables pour éviter les répétitions : les clients d'un côté, les produits d'un autre, les commandes qui les relient. Pour répondre à « quel chiffre d'affaires par ville ? », il faut **réunir** ces tables.

Deux opérations différentes existent :

- la **jointure** (`merge`) associe les lignes de deux tables qui partagent une clé : c'est `JOIN` en SQL ;
- la **concaténation** (`concat`) empile des tables de même structure, ou les juxtapose : c'est `UNION ALL`.

In [ ]:
import numpy as np
import pandas as pd

clients = pd.read_csv("../donnees/demo/clients.csv")
produits = pd.read_csv("../donnees/demo/produits.csv")
commandes = pd.read_csv("../donnees/demo/commandes.csv", parse_dates=["date_commande"])
commandes.shape, clients.shape, produits.shape

## 1. La jointure : `merge`

In [ ]:
cmd_prod = commandes.merge(produits, on="produit_id")
cmd_prod[["commande_id", "produit_id", "libelle", "prix", "quantite"]].head()

```sql
SELECT * FROM commandes JOIN produits ON commandes.produit_id = produits.produit_id;
```

Chaque commande reçoit les colonnes du produit correspondant. On peut maintenant calculer le montant :

In [ ]:
cmd_prod["montant"] = cmd_prod["quantite"] * cmd_prod["prix"]
cmd_prod.groupby("categorie")["montant"].sum().sort_values(ascending=False).head(3)

### Les quatre types de jointure

```
     inner              left               right              outer
   ┌───┬───┐         ┌───┬───┐          ┌───┬───┐          ┌───┬───┐
   │ A │▓B▓│ C       │▓A▓│▓B▓│ C        │ A │▓B▓│▓C▓│      │▓A▓│▓B▓│▓C▓│
   └───┴───┘         └───┴───┘          └───┴───┘          └───┴───┘
  clés communes    toutes les clés    toutes les clés     toutes les clés
                   de gauche          de droite           des deux côtés
```

Le paramètre `how` choisit le type ; la valeur par défaut est `"inner"`.

> **Prédiction.** La table `commandes` contient des identifiants clients 999 et 998, qui n'existent pas dans `clients`. Et quatre clients n'ont jamais commandé. Combien de lignes produira chacune des jointures suivantes ?

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    print(f"{how:6s}", commandes.merge(clients, on="client_id", how=how).shape)

- `inner` : seules les commandes dont le client existe ;
- `left` : toutes les commandes, avec `NaN` dans les colonnes client pour 999 et 998 ;
- `right` : tous les clients, avec `NaN` dans les colonnes commande pour ceux qui n'ont rien acheté ;
- `outer` : tout.

### Diagnostiquer une jointure : `indicator=True`

In [ ]:
diag = commandes.merge(clients, on="client_id", how="outer", indicator=True)
diag["_merge"].value_counts()

In [ ]:
diag.loc[diag["_merge"] == "left_only", "client_id"].unique()     # commandes sans client connu

In [ ]:
diag.loc[diag["_merge"] == "right_only", ["client_id", "prenom"]]  # clients sans commande

C'est le contrôle de qualité de base avant toute analyse : une clé orpheline signale une donnée manquante ou une erreur de saisie.

### Clés de noms différents, clés dans l'index

In [ ]:
fiches = clients.rename(columns={"client_id": "id"})
commandes.merge(fiches, left_on="client_id", right_on="id").filter(["commande_id", "client_id", "id", "prenom"]).head(3)

In [ ]:
clients_idx = clients.set_index("client_id")
commandes.merge(clients_idx, left_on="client_id", right_index=True).filter(["commande_id", "client_id", "prenom"]).head(3)

`left_index=True` et `right_index=True` utilisent l'index comme clé. Quand les deux tables sont indexées par la même clé, `df1.join(df2)` est un raccourci.

### Colonnes de même nom : `suffixes`

In [ ]:
a = pd.DataFrame({"produit_id": ["P01", "P02"], "prix": [89.9, 24.5]})
b = pd.DataFrame({"produit_id": ["P01", "P02"], "prix": [84.9, 24.5]})
a.merge(b, on="produit_id", suffixes=("_2025", "_2026"))

## 2. Cardinalité : quand une jointure multiplie les lignes

Une jointure associe **chaque** ligne de gauche à **chaque** ligne de droite qui a la même clé. Si la clé est répétée des deux côtés, les lignes se multiplient.

In [ ]:
tarifs = pd.DataFrame({"produit_id": ["P01", "P01", "P02"], "prix": [89.9, 79.9, 24.5],
                       "periode": ["normal", "soldes", "normal"]})
tarifs

> **Prédiction.** `commandes` contient combien de commandes du produit P01 ? Combien de lignes P01 obtiendra-t-on après la jointure avec `tarifs` ?

In [ ]:
print("commandes P01 :", (commandes["produit_id"] == "P01").sum())
fusion = commandes.merge(tarifs, on="produit_id")
print("lignes P01 après jointure :", (fusion["produit_id"] == "P01").sum())

Chaque commande de P01 apparaît deux fois, une par tarif. Un chiffre d'affaires calculé ensuite serait doublé pour ce produit, sans aucune erreur.

Le paramètre `validate` fait vérifier la cardinalité attendue par Pandas, qui lève une erreur si elle n'est pas respectée :

| `validate` | Signification |
|---|---|
| `"one_to_one"` | clé unique des deux côtés |
| `"many_to_one"` | clé unique à droite (cas commandes → produits) |
| `"one_to_many"` | clé unique à gauche |

In [ ]:
commandes.merge(tarifs, on="produit_id", validate="many_to_one")

In [ ]:
commandes.merge(produits, on="produit_id", validate="many_to_one").shape

## 3. Enchaîner les jointures

Une question peut traverser plusieurs tables. Dans la base Sakila (exercice du chapitre), un acteur est relié aux films par la table d'association `film_actor`.

In [ ]:
sakila = pd.read_excel("../donnees/bstorm/sakila.xlsx", sheet_name=None)
actor, film, film_actor = (sakila[n].drop(columns="Unnamed: 0") for n in ["actor", "film", "film_actor"])

films_par_acteur = (
    actor[["actor_id", "first_name", "last_name"]]
    .merge(film_actor[["actor_id", "film_id"]], on="actor_id")
    .merge(film[["film_id", "title"]], on="film_id")
)
films_par_acteur.head()

```sql
SELECT a.first_name, a.last_name, f.title
FROM actor a
JOIN film_actor fa ON a.actor_id = fa.actor_id
JOIN film f ON fa.film_id = f.film_id;
```

## 4. Semi-jointure et anti-jointure

Parfois, on ne veut pas les colonnes de l'autre table, seulement savoir si la clé y figure.

- **Semi-jointure** : les clients qui ont au moins une commande (`WHERE client_id IN (SELECT ...)`).
- **Anti-jointure** : les clients qui n'en ont aucune (`NOT IN`).

In [ ]:
ont_commande = clients[clients["client_id"].isin(commandes["client_id"])]
sans_commande = clients[~clients["client_id"].isin(commandes["client_id"])]
len(ont_commande), sans_commande["prenom"].tolist()

`isin` ne duplique jamais les lignes, contrairement à une jointure.

## 5. Une question ensembliste : « dans tous les… »

Quels clients ont commandé **par tous les canaux** existants ? C'est une *division relationnelle*, réputée difficile en SQL. En Pandas, on compte le nombre de valeurs distinctes par client et on le compare au nombre total :

In [ ]:
nb_canaux = commandes["canal"].nunique()
canaux_par_client = commandes.groupby("client_id")["canal"].nunique()
canaux_par_client[canaux_par_client == nb_canaux].index.tolist()

## 6. La concaténation : `concat`

### Empiler des lignes (`UNION ALL`)

In [ ]:
janvier = commandes[commandes["date_commande"].dt.month == 1]
fevrier = commandes[commandes["date_commande"].dt.month == 2]
deux_mois = pd.concat([janvier, fevrier])
len(janvier), len(fevrier), len(deux_mois)

Les colonnes sont alignées **par nom**. Une colonne absente d'un côté est remplie de `NaN`. Pour un `UNION` sans doublons, on ajoute `drop_duplicates()`.

`DataFrame.append` a été supprimé en Pandas 2 : on utilise toujours `pd.concat`, idéalement une seule fois sur une liste de morceaux plutôt qu'à chaque tour de boucle.

### L'index après concaténation

In [ ]:
s1 = pd.Series(["a", "b"])
s2 = pd.Series(["c", "d"])
empile = pd.concat([s1, s2])
empile

> **Prédiction.** Que renvoie `empile.loc[0]` ?

In [ ]:
empile.loc[0]

Les index d'origine sont conservés : l'étiquette 0 existe deux fois. `ignore_index=True` renumérote ; `keys=` garde la provenance dans un niveau d'index supplémentaire.

In [ ]:
pd.concat([s1, s2], ignore_index=True)

In [ ]:
pd.concat([s1, s2], keys=["source_1", "source_2"])

### Types mélangés

In [ ]:
pd.concat([pd.Series(["a", "b"]), pd.Series([0.5, 1.5])]).dtype

Une colonne qui reçoit du texte et des nombres prend le type `object`, lent et peu exploitable.

### Juxtaposer des colonnes (`axis=1`)

In [ ]:
s_qte = commandes["quantite"].head(3)
s_canal = commandes["canal"].head(3)
pd.concat([s_qte, s_canal], axis=1)

Avec `axis=1`, les morceaux sont **alignés sur l'index**, comme dans une jointure externe. Pour associer des tables par une colonne clé, `merge` reste l'outil adapté.

`to_frame` convertit une Series en DataFrame d'une colonne, utile avant une jointure ou un export :

In [ ]:
commandes.groupby("client_id")["commande_id"].count().to_frame("nb_commandes").head(3)

## 7. Application IA : enrichir une table d'observations

Pour un modèle qui prédit le comportement d'un client, chaque ligne doit décrire un client. On construit des variables à partir des commandes, puis on les joint à la table clients :

In [ ]:
variables = commandes.merge(produits, on="produit_id").assign(montant=lambda d: d["quantite"] * d["prix"])
variables = variables.groupby("client_id").agg(
    nb_commandes=("commande_id", "count"),
    ca_total=("montant", "sum"),
    derniere_commande=("date_commande", "max"),
)
table_ml = clients.merge(variables, on="client_id", how="left", validate="one_to_one")
table_ml.head()

La jointure `left` garde tous les clients, y compris ceux sans commande (`NaN`, à remplacer par 0 pour `nb_commandes`). `validate="one_to_one"` garantit qu'aucun client n'a été dupliqué.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| Clé non unique des deux côtés | lignes multipliées, totaux faux | `validate=`, et comparer `len` avant/après |
| Types de clés différents (`int` et `str`) | `ValueError` ou aucun résultat | aligner les types avec `astype` |
| `inner` par défaut | lignes perdues sans le savoir | choisir `how` explicitement, `indicator=True` |
| `pd.concat` dans une boucle | lenteur | accumuler dans une liste, concaténer une fois |
| `DataFrame.append` | `AttributeError` en Pandas 2 et 3 | `pd.concat` |
| Index dupliqués après `concat` | `loc` renvoie plusieurs lignes | `ignore_index=True` |

## Exercices

- Exercice Bstorm **Sakila**, questions **8 et 11 à 14**.
- Exercice supplémentaire **S8** : la jointure qui multiplie les lignes.
- Renforcement : drills **D05 Fictitious Names**, **D05 Auto MPG** (la solution d'origine utilise `append`), **D05 Housing Market**.

## À retenir

- `merge(on=..., how=...)` = `JOIN` ; `inner`, `left`, `right`, `outer`.
- `indicator=True` repère les orphelins ; `validate=` vérifie la cardinalité.
- Une clé répétée des deux côtés multiplie les lignes.
- `isin` pour les semi-jointures et anti-jointures.
- `pd.concat` empile (`UNION ALL`) ou juxtapose ; `ignore_index`, `keys`.